# Delivery Delay Prediction: EDA and Model Development

This notebook walks through the full workflow behind the app: load and clean the data, explore it, encode features, train **Logistic Regression** and **Random Forest**, compare them, and inspect the winner.

The same steps run as a script in `src/train_model.py`, which also saves the model used by the Streamlit app.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Make the project's src/ package importable when running from notebooks/
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src import visuals
from src.preprocessing import (FEATURE_COLUMNS, TARGET, build_preprocessor, clean_data,
                               describe_dataset, load_data)
from src.train_model import RANDOM_STATE, TEST_SIZE, evaluate, get_models

pd.set_option("display.max_columns", 30)

## 1. Load the dataset
If `data/delivery_data.csv` doesn't exist, it is generated automatically (6,000 synthetic orders).

In [ ]:
raw = load_data()
raw.head()

## 2. Dataset information

In [ ]:
describe_dataset(raw)

In [ ]:
raw.describe().T.round(2)

## 3. Handle missing values and duplicates

- **Duplicates** are removed here.
- **Missing feature values** are imputed *inside the model pipeline* (median for numbers, most frequent for categories). That way the imputation values are learned from the training data only, and the deployed model can handle incomplete inputs.

In [ ]:
df = clean_data(raw)
print(f"Rows before: {len(raw):,}  |  after removing duplicates: {len(df):,}")
df[FEATURE_COLUMNS].isna().sum().loc[lambda s: s > 0]

## 4. Exploratory data analysis

In [ ]:
visuals.plot_delivery_split(df)
plt.show()

In [ ]:
visuals.plot_delay_by_mode(df)
plt.show()
visuals.plot_delay_by_traffic(df)
plt.show()

In [ ]:
visuals.plot_delay_by_weather(df)
plt.show()
visuals.plot_distance_vs_delay(df)
plt.show()

In [ ]:
visuals.plot_delay_by_month(df)
plt.show()

**Takeaways:** delays rise steeply with distance, heavy traffic and storms; Economy shipping is the slowest mode; and the festive/monsoon months are the worst for on-time performance.

## 5. Encode features and split the data
Categorical columns are one-hot encoded and numeric columns are scaled inside a `ColumnTransformer`. The split is stratified so both sets keep the same delay rate.

In [ ]:
X, y = df[FEATURE_COLUMNS], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Delay rate  train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

encoded = build_preprocessor().fit_transform(X_train)
print(f"Feature matrix after encoding: {encoded.shape[1]} columns")

## 6. Train and compare models

In [ ]:
fitted, rows = {}, []
for name, model in get_models().items():
    pipe = Pipeline([("preprocessor", build_preprocessor()), ("model", model)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    rows.append({"Model": name, **evaluate(pipe, X_test, y_test)})

comparison = pd.DataFrame(rows)
comparison.round(4)

In [ ]:
visuals.plot_model_comparison(comparison)
plt.show()

best_name = comparison.loc[comparison["F1 Score"].idxmax(), "Model"]
print(f"Best model by F1 score: {best_name}")

## 7. Inspect the best model

In [ ]:
best = fitted[best_name]
pred = best.predict(X_test)
print(classification_report(y_test, pred, target_names=["On Time", "Delayed"]))

visuals.plot_confusion_matrix(confusion_matrix(y_test, pred, labels=[0, 1]))
plt.show()

In [ ]:
perm = permutation_importance(best, X_test, y_test, scoring="f1", n_repeats=8,
                              random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.DataFrame({"Feature": FEATURE_COLUMNS, "Importance": perm.importances_mean})
visuals.plot_feature_importance(importance)
plt.show()

## 8. Use the saved model
Run `python src/train_model.py` to (re)train and save `models/delivery_delay_model.pkl`. Below we load it and score one order.

In [ ]:
import joblib
from src.preprocessing import MODEL_PATH

if not MODEL_PATH.exists():
    from src.train_model import train_and_save
    train_and_save(verbose=False)

bundle = joblib.load(MODEL_PATH)
order = pd.DataFrame([{
    "Warehouse_Origin": "Mumbai", "Destination": "Chennai", "Product_Category": "Electronics",
    "Order_Value": 15000.0, "Distance_km": 1290.0, "Shipping_Mode": "Economy", "Customer_Type": "New",
    "Order_Day": "Saturday", "Order_Month": "November", "Weather_Condition": "Storm",
    "Traffic_Level": "High", "Number_of_Items": 3, "Previous_Delays": 3,
    "Processing_Time_hrs": 40.0, "Shipping_Cost": 150.0}], columns=FEATURE_COLUMNS)

prob = bundle["pipeline"].predict_proba(order)[0, 1]
print(f"Model: {bundle['model_name']}  |  Delay probability: {prob:.0%}  ->",
      "DELAYED" if prob >= 0.5 else "ON TIME")